# A first tutorial for `tensorium`: tensor fields on the 2-sphere


This notebook introduces the basic objects of the library through the example of the two-dimensional sphere $S^2$.


In [1]:
from sympy import simplify, symbols
from tensorium import *


## 1. Basics


We will use two stereographic charts:

- `X_N`, defined away from the north pole,
- `X_S`, defined away from the south pole.

On their overlap, the coordinate transformation is

$$
(u,v)=\left(\frac{x}{x^2+y^2},\frac{y}{x^2+y^2}\right),
$$

and the inverse has the same algebraic form.
### 1.1 Manifold, open sets and charts

We first create the abstract manifold, two open sets and two charts. The open sets represent the domains of the stereographic coordinates. We then create an atlas and attach it to the Manifold

In [2]:
# Abstract manifold
S2 = Manifold("S^2", 2)

# Chart domains
U_N = OpenSet("U_N", S2)  # S^2 without the north pole
U_S = OpenSet("U_S", S2)  # S^2 without the south pole

# Coordinate symbols used at construction time
x, y = symbols("x y", real=True)
u, v = symbols("u v", real=True)

# Two stereographic charts
X_N = Chart("X_N", U_N, (x, y))
X_S = Chart("X_S", U_S, (u, v), relations={X_N: (u/(u**2 + v**2), v/(u**2 + v**2))})

# Attach an atlas to the manifold
atlas = Atlas(S2, [X_N, X_S])
S2.set_atlas(atlas)

# CoordinateSymbol objects attached to each chart.
xN, yN = X_N.symbols
uS, vS = X_S.symbols

Display(S2)
Display(X_N)
Display(X_S)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Only one transition map between charts is provided explicitly. The atlas is treated as a graph: charts are nodes and known coordinate transformations are edges. When a change of chart is requested, the library searches for a shortest path between the two charts. If an edge is only known in the opposite direction, the library can try to invert the transformation symbolically and then compute the required Jacobian matrices when they are first needed.
</div>

### 1.2. Scalar fields


A scalar field is a tensor field of type $(0,0)$. Locally, it is represented by a single expression.
Below we define a scalar field by giving two local expressions, one in each stereographic chart:

$$
f_N(x,y)=\frac{1}{1+x^2+y^2},
\qquad
f_S(u,v)=\frac{u^2+v^2}{1+u^2+v^2}.
$$

Together, they describe the field on all of $S^2$. We will then verify that the two expressions agree on the overlap $U_N\cap U_S$.


In [3]:
f_N_local = LocalTensorField(X_N, (0, 0), 1/(1 + xN**2 + yN**2))
f_S_local = LocalTensorField(X_S, (0, 0), (uS**2 + vS**2)/(1 + uS**2 + vS**2))

Display(f_N_local, name="f")
Display(f_S_local, name="f")

f_N_to_S_overlap = f_N_local.transform_to(X_S)
f_S_on_overlap = f_S_local.restrict(f_N_to_S_overlap.open_set)

Display(f_N_to_S_overlap, name="f")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Display labels scalar fields as elements of <code>C^∞(M)</code>. However, it does not prove nor check that the expression is actually smooth on its whole domain. If you enter a non-smooth scalar field, the notation should not be interpreted as a validation.
</div>

In [4]:
f = TensorField(S2, (0, 0), {X_N: f_N_local, X_S: f_S_local}, ())

Display(f, name="f")

<IPython.core.display.Math object>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Scalar fields (and vector fields and one-forms) are specialized classes built on top of the general <code>TensorField</code> class. A tensor field may store several local representations, indexed by charts. This makes it possible to define an object on different open sets without forcing a single global coordinate expression. The user is responsible for providing enough local representations to cover the intended domain. The library also does not automatically prove compatibility on overlaps: if two formulas are given on intersecting chart domains, it will not certify by itself that transforming one representation tensorially gives the other.
</div>


### 1.3. Vector fields and one-forms


A vector field is a tensor field of type $(1,0)$, while a one-form is a tensor field of type $(0,1)$.

In coordinates, a vector field is written as
$
V = V^i \partial_i,
$
whereas a one-form is written as
$
\omega = \omega_i dx^i.
$
The library keeps track of the index variance: contravariant indices use `1`, covariant indices use `-1`.


In [5]:
# A rotation vector field, given directly in both charts.
V_N_local = LocalTensorField(X_N, (1, 0), [-yN, xN])
V_S_local = LocalTensorField(X_S, (1, 0), [-vS, uS])
V = Vector(S2, {X_N: V_N_local, X_S: V_S_local})

Display(V, name="V")


<IPython.core.display.Math object>

In [6]:
# The one-form omega = df, also given directly in both charts.
omega_N_local = LocalTensorField(X_N, (0, 1), [-2*xN/(1 + xN**2 + yN**2)**2, -2*yN/(1 + xN**2 + yN**2)**2])
omega_S_local = LocalTensorField(X_S, (0, 1), [2*uS/(1 + uS**2 + vS**2)**2, 2*vS/(1 + uS**2 + vS**2)**2])
omega = OneForm(S2, {X_N: omega_N_local, X_S: omega_S_local})

Display(omega, name=r"\omega")


<IPython.core.display.Math object>

One-forms can act on vector fields by contraction
$
\omega(V)=\omega_i V^i.
$

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Evaluation is performed chart by chart. If the one-form and the vector field already have a common local representation, the contraction is done there directly. If not, the library can use the atlas graph to transform one object to a compatible chart.
</div>


In [7]:
omega_on_V = omega(V)

Display(omega_on_V, name=r"\omega(V)")

<IPython.core.display.Math object>

Vector fields can also act on scalars by directional differentiation:
$
V(f)=V^i\partial_i f.
$

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
The output is again a global scalar field, with one local expression for every chart in which the operation can be evaluated.
</div>


In [8]:
Vf = V(f)

Display(Vf, name="V(f)")

<IPython.core.display.Math object>

The library can also compute the commutator of two vector fields and return another vector field: $[V,W]$


In [9]:
W_N_local = LocalTensorField(X_N, (1, 0), [xN, yN])
W_S_local = LocalTensorField(X_S, (1, 0), [-uS, -vS])
W = Vector(S2, {X_N: W_N_local, X_S: W_S_local})

bracket = V.commutator(W)
Display(bracket, name="[V,W]")


<IPython.core.display.Math object>

### 1.4. General tensor fields


General tensors are represented by their tensor type $(r,s)$ and by their index variance.
For example, the tensor product
$
T = V \otimes \omega
$
has type $(1,1)$. Its components are
$
T^i{}_j = V^i\omega_j.
$

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Since both factors were given in the two stereographic charts, the tensor product also has full local representations in both chart domains.
</div>


In [10]:
T = V * omega

Display(T, name="T")

<IPython.core.display.Math object>

Tensor fields of fixed type form a module over $C^\infty(M)$. Therefore, tensor fields of the same type can be added and subtracted, and they can also be multiplied by scalar fields. In addition, the library supports contractions between one contravariant index and one covariant index.

In [11]:
fT = f*T-T
trace_T = T.contraction(0, 1)


Display(fT, name="fT")
Display(trace_T, chart=X_N, name=r"T^i_i")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

A tensor can also be evaluated on rank-one tensor fields. For example, if $T$ has type $(1,1)$, then it naturally acts on one one-form and one vector field:

$$
T(\alpha, X)=T^i{}_j\,\alpha_i X^j.
$$

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
Tensor evaluation follows the same chart-selection logic as contractions and actions. 
The library first looks for charts common to the tensor and all its arguments. If no common chart is already available, it uses the atlas graph of known coordinate transformations and chooses a chart that minimizes the total number of chart changes needed.
</div>


In [12]:
T_on_omega_W = T(omega, W)

Display(T_on_omega_W, name=r"T(\omega,W)")

<IPython.core.display.Math object>

<div style="font-size:0.86em; line-height:1.42; padding:0.55em 0.80em; margin:0.55em 0 0.75em 0; border:1px solid rgba(148,163,184,0.45); border-radius:8px; background:rgba(148,163,184,0.08);">
The library also supports component symmetries. A tensor field can be given a symmetry dictionary telling the library which index tuples are equivalent and which representative component should be used for each equivalence class. Internally, this avoids storing every component independently: when a non-representative component is requested, the tensor redirects the query to the corresponding representative entry, possibly including the appropriate sign if the symmetry is antisymmetric.
</div>